# `ofr(series, start, end=None)` and `ofr_series()`

These functions give US short-term funding data and financial stress data from `indicators.ofr`, oldest first. The data comes from two products of the Office of Financial Research (OFR). The Short-term Funding Monitor has repo, money market funds, primary dealers, the New York Fed reference rates and Treasury yields. The Financial Stress Index is the second product.

`ofr_series()` gives the dataset, name, units, frequency and category of each series id. Each cell reads prod as `readonly_user`, and does not write. You can change the arguments and run a cell again.

Use the OFR mnemonic (`REPO-TRI_AR_OO-P`, `OFR_FSI`) or the OFR name of a series, in any case and spacing. The functions do not give days with no value, and do not fill them.

> **About the data shown.** Source: Office of Financial Research, "OFR Short-term Funding Monitor," refreshed daily, https://www.financialresearch.gov/short-term-funding-monitor/, and "OFR Financial Stress Index," https://www.financialresearch.gov/financial-stress-index/ (both accessed November 25, 2026). arora_data made one change: it divides the OFR dollar values by 1,000,000, thus they are in USD millions. All other values are the OFR values.
>
> OFR's disclaimer: "This OFR monitor is presented solely for informative purposes and should not be relied upon for financial decisions; it is not intended to provide any investment or financial advice." The OFR repo rates are "not intended to be referenced in contracts."
>
> Some of this data is not from OFR. The primary dealer statistics (`nypd`) are Federal Reserve Bank of New York data, under the Terms of Use at newyorkfed.org. The reference rates (`fnyr`: SOFR, TGCR, BGCR, EFFR, OBFR, with their percentiles and volumes) are New York Fed data, under its Reference Rates Terms of Use. The New York Fed is not responsible for publication of this data by arora_data, does not sanction or endorse any particular republication, and has no liability for your use.
>
> The Secured Overnight Financing Rate (SOFR) Data and Broad General Collateral Rate (BGCR) Data are calculated using data provided under a license granted to the New York Fed by DTCC Solutions LLC (“Solutions”), an affiliate of The Depository Trust & Clearing Corporation. Solutions, its affiliates, and third parties from which they obtained data have no liability for the content of this material.
>
> The Treasury yields (`tyld`) are US Treasury data, public domain. arora_data has no affiliation with the OFR or the New York Fed, and they do not endorse it. This notebook is not investment advice.

In [1]:
from datetime import date, timedelta
from decimal import Decimal

import polars as pl

from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.ofr import OfrSeries, OfrValue
from arora_data.sync import ofr, ofr_series


def side_by_side(ids, start):
    """The values of some series, one column for each series, joined on date."""
    frames = [to_df(ofr(i, start), OfrValue).select("date", pl.col("value").alias(n)) for i, n in ids.items()]
    joined = frames[0]
    for f in frames[1:]:
        joined = joined.join(f, on="date", how="full", coalesce=True)
    return joined.sort("date")

## The series
Each call reads the series from the table. The labels come from the latest row of each series. `first_date` and `latest_date` are the first and latest days that have a value. A series with no value on any day has no dates. `category` is the OFR dataset of the series.

The table also holds the OFR Hedge Fund Monitor. The functions do not give it until a person checks the terms of its SEC, CFTC, Fed and FICC sources. The number of series for each dataset, category, unit and frequency:

In [2]:
series = to_df(ofr_series(), OfrSeries)
print(series.height, "series")
series.group_by("dataset", "category", "units", "frequency") \
    .agg(pl.len().alias("series"), pl.col("first_date").min()).sort("dataset", "category", "units")

451 series


dataset,category,units,frequency,series,first_date
str,str,str,str,u32,date
"""ofr/financial_stress_index""","""fsi""","""index""","""daily""",9,2000-01-03
"""ofr/short_term_funding""","""fnyr""","""USD millions""","""daily""",5,2016-03-01
"""ofr/short_term_funding""","""fnyr""","""percent""","""daily""",25,2016-03-01
"""ofr/short_term_funding""","""mmf""","""USD millions""","""monthly""",26,2010-11-30
"""ofr/short_term_funding""","""mmf""","""percent""","""monthly""",16,2016-04-30
"""ofr/short_term_funding""","""nypd""","""USD millions""","""weekly""",194,2015-01-07
"""ofr/short_term_funding""","""repo""","""USD millions""","""daily""",96,2014-08-22
"""ofr/short_term_funding""","""repo""","""percent""","""daily""",68,2014-08-22
"""ofr/short_term_funding""","""tyld""","""percent""","""daily""",12,1990-01-02


The first two series of each category:

In [3]:
series.sort("series_id").group_by("category", maintain_order=True).head(2).select("series_id", "category", "name", "units")

series_id,category,name,units
str,str,str,str
"""FNYR-BGCR-A""","""fnyr""","""Broad General Collateral Rate""","""percent"""
"""FNYR-BGCR_1Pctl-A""","""fnyr""","""Broad General Collateral Rate:…","""percent"""
"""MMF-MMF_AG_TOT-M""","""mmf""","""Money Market Mutual Fund Inves…","""USD millions"""
"""MMF-MMF_BRA_TOT-M""","""mmf""","""Money Market Mutual Fund Inves…","""USD millions"""
"""NYPD-PD_AFtD_AG-A""","""nypd""","""Primary Dealer Aggregate Fails…","""USD millions"""
…,…,…,…
"""OFR_FSI_CREDIT""","""fsi""","""OFR FSI: credit""","""index"""
"""REPO-DVP_AR_B27-F""","""repo""","""DVP Service Average Rate: Term…","""percent"""
"""REPO-DVP_AR_B27-P""","""repo""","""DVP Service Average Rate: Term…","""percent"""


The series that have no value on any day:

In [4]:
series.filter(pl.col("latest_date").is_null()).select("series_id", "name")

series_id,name
str,str
"""NYPD-PD_SB_TOT-A""","""Primary Dealer Securities Borr…"
"""NYPD-PD_SL_GE30-A""","""Primary Dealer Securities Lent…"
"""NYPD-PD_SL_TOT-A""","""Primary Dealer Securities Lent…"
"""REPO-GCF_AR_B27-F""","""GCF Repo Service Average Rate:…"
"""REPO-GCF_AR_B27-P""","""GCF Repo Service Average Rate:…"
"""REPO-GCF_TV_B27-F""","""GCF Repo Service Transaction V…"
"""REPO-GCF_TV_B27-P""","""GCF Repo Service Transaction V…"


## One series, by id or by name
This is the tri-party repo overnight rate for the last two weeks, in percent. The id and the name, in any case and spacing, give the same rows. Each row is a business day. Weekends and holidays have no rows.

In [5]:
two_weeks = today_et() - timedelta(days=14)
assert ofr("REPO-TRI_AR_OO-P", two_weeks) == ofr("repo-tri_ar_oo-p", two_weeks) \
    == ofr(" tri-party AVERAGE rate:  overnight/open (preliminary) ", two_weeks)
to_df(ofr("REPO-TRI_AR_OO-P", two_weeks), OfrValue)

series_id,name,date,value,units
str,str,date,"decimal[38,8]",str
"""REPO-TRI_AR_OO-P""","""Tri-Party Average Rate: Overni…",2026-11-12,3.88000000,"""percent"""
"""REPO-TRI_AR_OO-P""","""Tri-Party Average Rate: Overni…",2026-11-13,3.88000000,"""percent"""
"""REPO-TRI_AR_OO-P""","""Tri-Party Average Rate: Overni…",2026-11-16,3.91000000,"""percent"""
"""REPO-TRI_AR_OO-P""","""Tri-Party Average Rate: Overni…",2026-11-17,3.91000000,"""percent"""
"""REPO-TRI_AR_OO-P""","""Tri-Party Average Rate: Overni…",2026-11-18,3.89000000,"""percent"""
"""REPO-TRI_AR_OO-P""","""Tri-Party Average Rate: Overni…",2026-11-19,3.89000000,"""percent"""
"""REPO-TRI_AR_OO-P""","""Tri-Party Average Rate: Overni…",2026-11-20,3.90000000,"""percent"""


## The three repo rates
These are the DVP (bilateral, centrally cleared), GCF (general collateral, centrally cleared) and tri-party overnight rates, in percent. A day with no value in one segment shows `null`. The tri-party rate comes one business day after the other two, thus its latest day is often `null`.

In [6]:
side_by_side({"REPO-DVP_AR_OO-P": "DVP", "REPO-GCF_AR_AG-P": "GCF", "REPO-TRI_AR_OO-P": "tri-party"}, two_weeks)

date,DVP,GCF,tri-party
date,"decimal[38,8]","decimal[38,8]","decimal[38,8]"
2026-11-12,3.92000000,3.95000000,3.88000000
2026-11-13,3.91000000,3.94000000,3.88000000
2026-11-16,3.94000000,3.96000000,3.91000000
2026-11-17,3.94000000,3.97000000,3.91000000
2026-11-18,3.92000000,3.95000000,3.89000000
2026-11-19,3.92000000,3.96000000,3.89000000
2026-11-20,3.93000000,3.94000000,3.90000000
2026-11-23,3.93000000,3.96000000,null


## Units
Rates are in percent: 3.91 is 3.91%. Dollar series are in USD millions: 3951744.1 is approximately $3.95 trillion. These are the latest values of tri-party volume and money market fund holdings, in USD trillions:

In [7]:
for i in ["REPO-TRI_TV_TOT-P", "MMF-MMF_TOT-M"]:
    v = ofr(i, today_et() - timedelta(days=62))[-1]
    print(f"{v.name:<52} {v.date}  {(v.value / Decimal(10**6)).quantize(Decimal('0.001')):>7}")

Tri-Party Transaction Volume: Total (Preliminary)    2026-11-20    2.289


Money Market Mutual Fund Investments: Total          2026-10-31    8.473


## Money market fund holdings, monthly
Each month has one value, with the last calendar day of the month as its date. The holdings at the last month of each year, in USD trillions:

In [8]:
mmf = to_df(ofr("MMF-MMF_TOT-M", date(1900, 1, 1)), OfrValue)
print(f"{mmf.height} months, {mmf['date'].min()} to {mmf['date'].max()}")
(mmf.group_by(pl.col("date").dt.year().alias("year")).agg(pl.col("value").sort_by("date").last())
    .with_columns((pl.col("value") / 1_000_000).round(2).alias("USD trillions")).drop("value").sort("year"))

192 months, 2010-11-30 to 2026-10-31


year,USD trillions
i32,"decimal[38,8]"
2010,3.04000000
2011,2.91000000
2012,2.98000000
2013,3.05000000
2014,3.08000000
…,…
2022,5.21000000
2023,6.40000000
2024,7.25000000


## The reference rates and the Treasury yields
`fnyr` has the OFR copy of the New York Fed reference rates, with their percentiles and volumes. `tyld` has the OFR copy of the Treasury constant maturity yields. `nyfed()` and `treasury()` give the data from the New York Fed and Treasury. SOFR and the 10-year yield for the last two weeks, in percent:

In [9]:
side_by_side({"FNYR-SOFR-A": "SOFR", "TYLD-TCMR-10Yr-A": "10-year yield"}, two_weeks)

date,SOFR,10-year yield
date,"decimal[38,8]","decimal[38,8]"
2026-11-12,3.90000000,5.10000000
2026-11-13,3.90000000,5.15000000
2026-11-16,3.93000000,5.12000000
2026-11-17,3.92000000,5.10000000
2026-11-18,3.90000000,5.07000000
2026-11-19,3.91000000,5.04000000
2026-11-20,3.92000000,4.92000000
2026-11-23,3.92000000,4.92000000


## Days with no OFR value
OFR gives no value for a day with no trades, or for a day that fails its disclosure threshold. The threshold keeps business-confidential data private. The table stores these days as NULL, and `ofr()` does not give them. The DVP overnight rate has 96 such days, from 2018-05-07 to 2025-02-06. The number of weekdays in each year with a GCF value and no DVP value:

In [10]:
gcf = to_df(ofr("REPO-GCF_AR_AG-P", date(1900, 1, 1)), OfrValue).select("date")
dvp = to_df(ofr("REPO-DVP_AR_OO-P", date(1900, 1, 1)), OfrValue)
gcf.join(dvp, on="date", how="anti").group_by(pl.col("date").dt.year().alias("year")).len().sort("year")

year,len
i32,u32
2018,8
2019,6
2020,16
2021,13
2022,16
2023,12
2024,22
2025,3


The tri-party rate also has no row on 2022-06-17. The tri-party volume has a row on that day:

In [11]:
day = date(2022, 6, 17)
print("rate:", ofr("REPO-TRI_AR_OO-P", day, day), " volume:", [v.value for v in ofr("REPO-TRI_TV_TOT-P", day, day)])

rate: []  volume: [Decimal('3414528.21919511')]


## Rates below zero
DVP was below zero on 94 days, from 2020-03-24 to 2022-03-14. These are the OFR values, and the functions give them as the table stores them. The days below zero in each year:

In [12]:
dvp.filter(pl.col("value") < 0).group_by(pl.col("date").dt.year().alias("year")) \
    .agg(pl.len().alias("days"), pl.col("value").min().alias("lowest")).sort("year")

year,days,lowest
i32,u32,"decimal[38,8]"
2020,9,-0.02000000
2021,83,-0.08000000
2022,2,-0.02000000


## The Financial Stress Index
The index is daily from 2000-01-03, in units "index". A value of 0 is average stress. A value above 0 is more stress, and a value below 0 is less stress. The index is the sum of its five categories (credit, equity valuation, safe assets, funding, volatility), with a maximum difference of 0.002 from OFR rounding. The sum of its three regions (United States, other advanced economies, emerging markets) is only approximately equal to the index, with a maximum difference of 0.108. The last two weeks:

In [13]:
fsi = {s.series_id: s.series_id.removeprefix("OFR_FSI_").lower() if s.series_id != "OFR_FSI" else "index"
       for s in ofr_series() if s.category == "fsi"}
side_by_side(fsi, two_weeks)

date,index,credit,emerging,equity_valuation,funding,other_advanced,safe_assets,us,volatility
date,"decimal[38,8]","decimal[38,8]","decimal[38,8]","decimal[38,8]","decimal[38,8]","decimal[38,8]","decimal[38,8]","decimal[38,8]","decimal[38,8]"
2026-11-11,-1.95000000,-0.90400000,-0.48100000,-0.45400000,-0.22000000,-0.31300000,-0.30100000,-1.15500000,-0.07100000
2026-11-12,-1.88000000,-0.92100000,-0.47300000,-0.44400000,-0.19500000,-0.31600000,-0.29400000,-1.09100000,-0.02600000
2026-11-13,-1.87400000,-0.92400000,-0.47200000,-0.43900000,-0.19100000,-0.34500000,-0.29100000,-1.05600000,-0.02800000
2026-11-16,-1.55500000,-0.90400000,-0.45600000,-0.40500000,-0.12800000,-0.26200000,-0.28900000,-0.83700000,0.17100000
2026-11-17,-1.12900000,-0.87300000,-0.45000000,-0.37700000,-0.05300000,-0.03500000,-0.29000000,-0.64400000,0.46300000
2026-11-18,-1.38700000,-0.85300000,-0.44700000,-0.36800000,-0.12200000,-0.20300000,-0.28400000,-0.73700000,0.24000000
2026-11-19,-1.25900000,-0.84100000,-0.44000000,-0.34300000,-0.12200000,-0.25900000,-0.28100000,-0.56100000,0.32800000
2026-11-20,-0.44900000,-0.78200000,-0.42100000,-0.28400000,-0.01600000,0.05900000,-0.27400000,-0.08700000,0.90600000


The five highest days are all in October 2008:

In [14]:
to_df(ofr("OFR_FSI", date(1900, 1, 1)), OfrValue).sort("value", descending=True).head(5).select("date", "value")

date,value
date,"decimal[38,8]"
2008-10-10,29.32000000
2008-10-27,29.22600000
2008-10-24,28.29400000
2008-10-28,27.23500000
2008-10-13,26.95900000


## Revisions
The main repo reads the full history of each dataset on each run. Thus the OFR revisions replace the earlier values, and the table does not keep the earlier values.

## Bad input
An unknown id or name, or a start after the end, raises a clear error before the function reads values. A Hedge Fund Monitor id is unknown to `ofr()` at this time.

In [15]:
for args in [("sofr", date(2026, 1, 1)), ("FPF-ALLQHF_ALTERNATE_COUNT", date(2026, 1, 1)), (None, date(2026, 1, 1)),
             ("REPO-DVP_AR_OO-P", date(2026, 1, 2), date(2026, 1, 1))]:
    try:
        ofr(*args)
    except ValueError as e:
        print(e)

'sofr' is not a stored OFR series: use an id or name from ofr_series(), such as OFR_FSI or 'OFR Financial Stress Index'


'FPF-ALLQHF_ALTERNATE_COUNT' is not a stored OFR series: use an id or name from ofr_series(), such as OFR_FSI or 'OFR Financial Stress Index'


None is not a stored OFR series: use an id or name from ofr_series(), such as OFR_FSI or 'OFR Financial Stress Index'
start must be on or before end
